# Certainty Labs -- End-to-End Demo

**Constraint-Guaranteed Outputs for Production AI**

This notebook demonstrates the full pipeline:
1. Define constraints in YAML
2. Generate labeled training data (or use default EORM GSM8K-Llama dataset)
3. Train a TransEBM energy model (matching EORM architecture)
4. Rerank LLM candidates by energy score

**Training loop matches [EnergyORM](https://github.com/ericjiang18/EnergyORM)** -- designed for Kaggle T4 GPU.

In [ ]:
# Install dependencies (run once)
!pip install -q torch transformers accelerate pydantic pyyaml jsonlines tqdm plotly

In [ ]:
import torch
import json
import os
import sys

# GPU detection
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
    print(f'FP16 supported: {torch.cuda.is_bf16_supported()}')
else:
    print('No GPU -- training will be slow on CPU')

## Step 1: Define Constraints

In [ ]:
from certainty.compiler.compiler import ConstraintCompiler

yaml_content = """
name: portfolio_balance
version: '1.0'
description: Portfolio weights must sum to 1.0, all non-negative, max 40% per asset
output_format: json
parse_error_penalty: 100.0
rules:
  - type: sum_equals
    field: weights.*
    equals: 1.0
    tolerance: 0.001
    penalty_weight: 10.0
  - type: range
    field: weights.*
    min: 0.0
    max: 0.40
    penalty_weight: 5.0
  - type: count
    field: weights
    at_most: 5
    penalty_weight: 2.0
"""

compiler = ConstraintCompiler()
energy_fn = compiler.compile_yaml(yaml_content)
print(f'Compiled: {energy_fn._config.name}')

# Test it
valid = {'weights': {'AAPL': 0.33, 'GOOG': 0.33, 'MSFT': 0.34}}
invalid = {'weights': {'AAPL': 0.5, 'GOOG': 0.5, 'MSFT': 0.5}}
print(f'Valid portfolio energy: {energy_fn(valid):.4f}')
print(f'Invalid portfolio energy: {energy_fn(invalid):.4f}')

## Step 2: Generate Training Data

In [ ]:
from certainty.data.sampler import MockSampler
from certainty.data.generator import DataGenerator, GeneratorConfig

sampler = MockSampler(domain='portfolio')
gen_config = GeneratorConfig(n_samples=500)
gen = DataGenerator(energy_fn, sampler, gen_config)

os.makedirs('certainty_workspace', exist_ok=True)
stats = gen.generate_eorm_format(
    prompt='Generate a valid portfolio allocation as JSON.',
    output_path='certainty_workspace/training_data.jsonl',
)
print(f'Stats: {stats}')

## Step 3: Train TransEBM

In [ ]:
from certainty.models.trainer import EBMTrainer, TrainingConfig

config = TrainingConfig(
    epochs=20,
    d_model=256,            # smaller for demo speed
    n_layers=2,
    n_heads=4,
    lr=5e-5,
    max_length=512,         # sufficient for portfolio JSON
    validate_every=1,       # validate every epoch (matching EORM)
)

trainer = EBMTrainer(config)
print(f'Device: {trainer.device}')
print(f'FP16 AMP: {trainer.use_amp}')

# Uses certainty_workspace/training_data.jsonl if provided,
# otherwise falls back to default EORM demo dataset
# (demo_dataset/results_gsm8k_llama_train_n4_temp0.7_p0.9_train_corrected.jsonl)
metrics = trainer.train(
    data_path='certainty_workspace/training_data.jsonl',
    output_dir='certainty_workspace/model',
)

print(f'\nBest validation accuracy: {metrics["best_val_acc"]:.1f}%')
print(f'Training time: {metrics["elapsed_seconds"]:.0f}s')

In [ ]:
# Plot training curves
import plotly.graph_objects as go
from plotly.subplots import make_subplots

history = metrics['history']
fig = make_subplots(rows=1, cols=2, subplot_titles=('Training Loss', 'Validation Accuracy'))

fig.add_trace(
    go.Scatter(y=[h['loss'] for h in history], mode='lines+markers',
               line=dict(color='#6C5CE7', width=2)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(y=[h['val_acc'] for h in history], mode='lines+markers',
               line=dict(color='#00b894', width=2)),
    row=1, col=2,
)
fig.update_layout(height=350, showlegend=False, template='plotly_dark')
fig.update_xaxes(title_text='Epoch')
fig.show()

## Step 4: Rerank Candidates

In [ ]:
from certainty.inference.reranker import ConstraintReranker

reranker = ConstraintReranker(
    model_path=metrics['model_path'],
    tokenizer_path=metrics['tokenizer_path'],
)

# Generate some test candidates
candidates = sampler.sample('portfolio', n=20)
best, best_idx, energies = reranker.rerank(candidates)

print(f'Best candidate (#{best_idx}): {best}')
print(f'Best energy: {energies[best_idx]:.4f}')

# Check violation rates
violations_before = sum(1 for c in candidates if energy_fn(json.loads(c)) > 0.01)
best_violation = energy_fn(json.loads(best)) > 0.01

print(f'\nBaseline violation rate: {violations_before}/{len(candidates)} = {100*violations_before/len(candidates):.1f}%')
print(f'Selected candidate violates: {best_violation}')

In [ ]:
# Energy distribution
import numpy as np

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f'#{i+1}' for i in np.argsort(energies)],
    y=[energies[i] for i in np.argsort(energies)],
    marker_color=['#00b894' if i == best_idx else '#636e72' for i in np.argsort(energies)],
))
fig.update_layout(
    title='Candidate Energy Scores (lower = better)',
    xaxis_title='Candidate',
    yaxis_title='Energy',
    template='plotly_dark',
    height=350,
)
fig.show()